# Introducción a NumPy

<div style="height: 4px; background-color: #4C78A8; border-radius: 2px; margin: 0.8em 0 1.2em;"></div>

<div style="padding: 0.8em 1em; margin: 0.8em 0 1em; border-left: 4px solid #4C78A8; background-color: rgba(76,120,168,0.08); border-radius: 4px; line-height: 1.6;">
<p style="margin: 0.45em 0;">Un <strong>array</strong> o <strong>arreglo</strong> organiza datos en una o varias dimensiones. Podemos imaginar una serie de temperaturas como una fila de valores, una tabla de estaciones y días como una estructura de dos dimensiones, y una sucesión de mapas como una estructura de tres dimensiones. En NumPy, estos objetos pertenecen a la clase <code>ndarray</code>.</p>
</div>

**NumPy** es una biblioteca de Python dedicada al cálculo numérico. Permite trabajar con arrays y aplicar operaciones matemáticas, estadísticas y de álgebra lineal sobre conjuntos de valores. También incluye herramientas para generar números aleatorios y calcular transformadas de Fourier, y se integra con bibliotecas como pandas, Matplotlib y SciPy.

Su interés no consiste únicamente en acelerar los cálculos. Expresiones como `temperaturas + 2` permiten operar sobre una colección completa sin escribir un bucle para cada elemento. Esta forma de trabajo se conoce como **vectorización**.

En meteorología y ciencias del clima podemos utilizarla para analizar series de observaciones o campos de temperatura, presión y precipitación. Los ejemplos siguientes emplean datos sintéticos o valores elegidos con fines didácticos.

Importamos la biblioteca con `import numpy as np`. El nombre abreviado `np` es una convención que permite reconocer sus funciones, como `np.array()` o `np.mean()`.
<p style="font-size: 0.9em; margin: 0.8em 0 0.3em;"><a href="https://numpy.org/doc/stable/user/absolute_beginners.html">Introducción oficial a NumPy</a></p>

In [45]:
import time

import numpy as np

## 0. Arreglos de NumPy frente a listas de Python

Comparemos el cálculo de una media con una lista y con un array que contienen los mismos números. Utilizamos un millón de enteros consecutivos para que la medición resulte más informativa. Son valores de prueba, no temperaturas observadas.

La función auxiliar `cronometra()` repite cada cálculo y devuelve el resultado junto con la mediana de los tiempos medidos. La creación de los datos queda fuera del cronometraje. Las expresiones `lambda` permiten pasarle una operación sin ejecutarla todavía; será la propia función la que la ejecute durante la medición.

In [46]:
# Ambas estructuras contienen los enteros de 0 a N - 1.
N = 1_000_000
numeros_lista = list(range(N))
numeros_array = np.arange(N, dtype=np.int64)


def cronometra(funcion, descripcion: str, repeticiones: int = 7):
    """Devuelve el resultado y el tiempo mediano de una operación, en ms.

    `funcion` debe poder llamarse sin argumentos. Se ejecuta una vez
    antes de medir y después tantas veces como indique `repeticiones`.
    La operación debe ser repetible, sin modificar sus datos de entrada.
    """
    if repeticiones < 1:
        raise ValueError("El número de repeticiones debe ser positivo.")

    # La primera llamada evita que la medición dependa solo del arranque.
    resultado = funcion()
    duraciones_ms = []
    for _ in range(repeticiones):
        inicio = time.perf_counter()
        resultado = funcion()
        duraciones_ms.append((time.perf_counter() - inicio) * 1_000)

    tiempo_mediano_ms = float(np.median(duraciones_ms))
    print(f"{descripcion} — mediana de {repeticiones} medidas: "
          f"{tiempo_mediano_ms:.3f} ms")
    return resultado, tiempo_mediano_ms


media_lista, t_lista = cronometra(
    lambda: sum(numeros_lista) / len(numeros_lista),
    "Media con lista (sum/len)",
)
media_array, t_array = cronometra(
    lambda: numeros_array.mean(),
    "Media con array (.mean())",
)

print(f"\nMedia con lista: {media_lista:.1f}")
print(f"Media con array: {media_array:.1f}")
print(f"Cociente de tiempos lista/array: {t_lista / t_array:.2f}")

Media con lista (sum/len) — mediana de 7 medidas: 12.962 ms
Media con array (.mean()) — mediana de 7 medidas: 2.268 ms

Media con lista: 499999.5
Media con array: 499999.5
Cociente de tiempos lista/array: 5.72


El cociente compara los tiempos de esta ejecución. Si es mayor que uno, el cálculo con la lista ha tardado más; si es menor, ha ocurrido lo contrario. No es una propiedad fija de NumPy ni permite anticipar el resultado de cualquier otra operación.

NumPy suele ser eficaz al procesar muchos valores numéricos del mismo tipo mediante rutinas compiladas. Esto no significa que deje de recorrer los datos. Tampoco `sum()` ejecuta necesariamente un bucle escrito en Python, por lo que la comparación depende de las operaciones, los tipos de datos y el equipo utilizado.

<div style="padding: 0.8em 1em; margin: 0.8em 0 1em; border-left: 4px solid #59A14F; background-color: rgba(89,161,79,0.08); border-radius: 4px; line-height: 1.6;">
<p style="margin: 0.45em 0;">Las listas siguen siendo útiles como colecciones generales. La elección depende de la tarea. Cuando los datos ya están en un array y necesitamos operar sobre muchos elementos, la vectorización permite expresar el cálculo de forma clara y suele ser eficiente. Una medición aislada no sustituye a comprobar el rendimiento en el problema real.</p>
</div>

## 1. Fundamentos de los arreglos de NumPy

Comenzaremos creando arrays a partir de listas y tuplas. Después veremos cómo reconocer su número de dimensiones, su forma y su tipo de dato, y cómo generar secuencias o estructuras de tamaño conocido.

### 1.1 Crear arrays desde listas o tuplas

Una lista puede reunir objetos de distintos tipos. Un array numérico de NumPy utiliza un **tipo de dato común**, indicado por `dtype`, lo que facilita aplicar cálculos al conjunto. Para construirlo a partir de una lista o una tupla utilizamos `np.array()`.

Una colección simple produce un array de una dimensión. Una lista de listas de igual longitud permite formar una tabla de dos dimensiones. En el ejemplo, cada fila corresponde a una estación y cada columna a un día.

<div style="overflow-x: auto; margin: 0.8em 0 1.2em;">
<table style="border-collapse: collapse; width: 100%; font-size: 0.96em; line-height: 1.5; text-align: left; table-layout: fixed;">
<colgroup><col style="width: 22%;"><col style="width: 78%;"></colgroup>
<thead><tr style="background-color: rgba(76,120,168,0.10);">
<th scope="col" style="padding: 0.6em 0.8em; text-align: left; border-bottom: 2px solid rgba(114,160,193,0.55);">Atributo</th>
<th scope="col" style="padding: 0.6em 0.8em; text-align: left; border-bottom: 2px solid rgba(114,160,193,0.55);">Qué nos indica</th>
</tr></thead>
<tbody>
<tr>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;"><code>ndim</code></td>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;">El número de dimensiones o <strong>ejes</strong>. Una serie tiene un eje; una tabla de estaciones y días tiene dos.</td>
</tr>
<tr>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;"><code>shape</code></td>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;">El tamaño de cada eje, expresado como una tupla. La forma <code>(3, 4)</code> representa tres filas y cuatro columnas.</td>
</tr>
<tr>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;"><code>size</code></td>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;">El número total de elementos. Para la forma <code>(3, 4)</code>, es <code>3 × 4 = 12</code>.</td>
</tr>
<tr>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;"><code>dtype</code></td>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;">El tipo de dato de los elementos, como <code>float64</code> o <code>int32</code>. Influye en los valores representables y en la memoria necesaria.</td>
</tr>
<tr>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;"><code>nbytes</code></td>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;">Los bytes correspondientes a los elementos del array. No incluye toda la información auxiliar del objeto.</td>
</tr>
</tbody></table>
</div>

Los atributos se consultan sin paréntesis, como en `temperaturas_diarias.shape`. En cambio, un método realiza una operación y se llama con paréntesis, como `temperaturas_diarias.mean()`.

<div style="padding: 0.8em 1em; margin: 0.8em 0 1em; border-left: 4px solid #59A14F; background-color: rgba(89,161,79,0.08); border-radius: 4px; line-height: 1.6;">
<p style="margin: 0.45em 0;">Una forma como <code>(7,)</code> corresponde a un array de un solo eje, no a una tabla con una fila. La coma forma parte de la representación de una tupla con un único elemento. NumPy tampoco conoce por sí mismo qué eje representa estaciones, días o niveles; debemos mantener esa correspondencia al programar.</p>
</div>

Para datos grandes puede resultar útil convertir `float64` a `float32`. Cada elemento pasa de ocupar ocho bytes a cuatro, pero también se reduce la precisión. En el ejemplo usamos `astype()` para crear esa versión sin modificar el array original.
<p style="font-size: 0.9em; margin: 0.8em 0 0.3em;"><a href="https://numpy.org/doc/stable/reference/generated/numpy.ndarray.nbytes.html">Atributos y memoria de un array</a></p>

In [47]:
# Temperaturas máximas diarias sintéticas de una semana, en °C.
temperaturas_diarias = np.array([18.2, 19.7, 20.1, 19.5, 21.3, 22.0, 20.8])

print("Temperaturas diarias (°C):", temperaturas_diarias)
print("Número de dimensiones:", temperaturas_diarias.ndim)
print("Forma:", temperaturas_diarias.shape)
print("Número de elementos:", temperaturas_diarias.size)
print("Tipo de dato:", temperaturas_diarias.dtype)

# Cada fila contiene cuatro días de una estación diferente.
temperaturas_rejilla = np.array([
    [17.5, 18.1, 19.0, 18.6],  # Estación costera
    [15.2, 15.8, 16.4, 16.0],  # Estación de valle
    [20.1, 20.5, 21.2, 20.9],  # Estación urbana
])
print("\nTabla de estaciones × días (°C):\n", temperaturas_rejilla)
print("Forma:", temperaturas_rejilla.shape)
print("Número de elementos:", temperaturas_rejilla.size)

# La conversión crea otro array; el original conserva su tipo y sus datos.
temperaturas_float32 = temperaturas_rejilla.astype(np.float32)
print("\nBytes de los datos en float64:", temperaturas_rejilla.nbytes)
print("Bytes de los datos en float32:", temperaturas_float32.nbytes)

Temperaturas diarias (°C): [18.2 19.7 20.1 19.5 21.3 22.  20.8]
Número de dimensiones: 1
Forma: (7,)
Número de elementos: 7
Tipo de dato: float64

Tabla de estaciones × días (°C):
 [[17.5 18.1 19.  18.6]
 [15.2 15.8 16.4 16. ]
 [20.1 20.5 21.2 20.9]]
Forma: (3, 4)
Número de elementos: 12

Bytes de los datos en float64: 96
Bytes de los datos en float32: 48


### 1.2 Funciones de inicialización frecuentes

No siempre necesitamos escribir cada valor. NumPy permite construir secuencias regulares, repartir un número de puntos entre dos límites o crear arrays rellenos con un valor inicial.

<div style="overflow-x: auto; margin: 0.8em 0 1.2em;">
<table style="border-collapse: collapse; width: 100%; font-size: 0.96em; line-height: 1.5; text-align: left; table-layout: fixed;">
<colgroup><col style="width: 42%;"><col style="width: 58%;"></colgroup>
<thead><tr style="background-color: rgba(76,120,168,0.10);">
<th scope="col" style="padding: 0.6em 0.8em; text-align: left; border-bottom: 2px solid rgba(114,160,193,0.55);">Función</th>
<th scope="col" style="padding: 0.6em 0.8em; text-align: left; border-bottom: 2px solid rgba(114,160,193,0.55);">Uso habitual</th>
</tr></thead>
<tbody>
<tr>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;"><code>np.arange(inicio, fin, paso)</code></td>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;">Genera una secuencia con un paso fijo. El límite final queda excluido en los ejemplos con enteros.</td>
</tr>
<tr>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;"><code>np.linspace(inicio, fin, num)</code></td>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;">Distribuye <code>num</code> valores equiespaciados. Por defecto incluye ambos extremos.</td>
</tr>
<tr>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;"><code>np.zeros(shape)</code></td>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;">Crea un array de ceros con la forma indicada.</td>
</tr>
<tr>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;"><code>np.ones(shape)</code></td>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;">Crea un array de unos.</td>
</tr>
<tr>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;"><code>np.full(shape, valor)</code></td>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;">Rellena el array con un valor constante.</td>
</tr>
</tbody></table>
</div>

Conviene elegir `arange` cuando importa el **paso**, y `linspace` cuando importa el **número de puntos** o incluir los extremos. Con pasos decimales, `arange` puede verse afectado por el redondeo de los números de coma flotante.

Para generar datos aleatorios creamos un generador con `rng = np.random.default_rng(42)`. El número `42` es una **semilla**, que fija su estado inicial. `rng.random((2, 3))` genera valores uniformes entre cero, incluido, y uno, excluido; `rng.standard_normal((2, 3))` genera una muestra de una distribución normal de media cero y desviación estándar uno.

<div style="padding: 0.8em 1em; margin: 0.8em 0 1em; border-left: 3px solid #72A0C1; background-color: rgba(76,120,168,0.025); border-radius: 4px; line-height: 1.6;">
<p style="margin: 0.45em 0;">Al crear de nuevo el generador con la misma semilla y repetir las mismas llamadas en el mismo entorno, obtenemos los mismos valores. Las llamadas sucesivas sobre un generador existente hacen avanzar la secuencia. La media y la desviación de una muestra pequeña no tienen por qué coincidir con las de la distribución utilizada.</p>
</div>

En código anterior también encontrarás `np.random.seed()`, `np.random.rand()` y `np.random.randn()`. Esa interfaz sigue disponible; aquí utilizamos un generador explícito para no depender de un estado aleatorio global.

<div style="padding: 0.8em 1em; margin: 0.8em 0 1em; border-left: 4px solid #C58A32; background-color: rgba(197,138,50,0.08); border-radius: 4px; line-height: 1.6;">
<p style="margin: 0.45em 0;">Un cero es un valor numérico válido, no una marca de dato ausente. Para representar observaciones desconocidas en un array de coma flotante podemos utilizar <code>np.nan</code>, que retomaremos en el apartado de estadística.</p>
</div>
<p style="font-size: 0.9em; margin: 0.8em 0 0.3em;"><a href="https://numpy.org/doc/stable/reference/random/generator.html">Generación de números aleatorios</a> · <a href="https://numpy.org/doc/stable/reference/generated/numpy.linspace.html">Secuencias con linspace</a></p>

In [48]:
# Horas de un día cada tres horas; el límite 24 queda excluido.
horas = np.arange(0, 24, 3)
print("Horas cada 3 h:", horas)

# Siete puntos desde el polo sur al norte, incluidos ambos extremos.
latitudes = np.linspace(-90, 90, 7)
print("Latitudes (grados):", latitudes)

matriz_ceros = np.zeros((2, 3))
vector_unos = np.ones(5)
matriz_presion = np.full((2, 2), 1013.25)  # Valor constante de ejemplo, en hPa.

print("\nMatriz de ceros:\n", matriz_ceros)
print("Vector de unos:", vector_unos)
print("Matriz de presión (hPa):\n", matriz_presion)

# Recrear el generador en esta celda permite repetir estas dos muestras.
rng_inicializacion = np.random.default_rng(42)
ruido_uniforme = rng_inicializacion.random((2, 3))
ruido_normal = rng_inicializacion.standard_normal((2, 3))

print("\nMuestra uniforme en [0, 1):\n", ruido_uniforme)
print("Muestra de una normal estándar:\n", ruido_normal)

Horas cada 3 h: [ 0  3  6  9 12 15 18 21]
Latitudes (grados): [-90. -60. -30.   0.  30.  60.  90.]

Matriz de ceros:
 [[0. 0. 0.]
 [0. 0. 0.]]
Vector de unos: [1. 1. 1. 1. 1.]
Matriz de presión (hPa):
 [[1013.25 1013.25]
 [1013.25 1013.25]]

Muestra uniforme en [0, 1):
 [[0.77395605 0.43887844 0.85859792]
 [0.69736803 0.09417735 0.97562235]]
Muestra de una normal estándar:
 [[ 0.1278404  -0.31624259 -0.01680116]
 [-0.85304393  0.87939797  0.77779194]]


## 2. Propiedades y operaciones básicas con arrays

Una vez creado un array, podemos consultar valores concretos, seleccionar una parte de los datos o combinarlo con otros arrays. Trabajaremos con tablas de estaciones y días para reconocer qué representa cada eje.

### 2.1 Indexación y slicing multidimensional

La **indexación** permite acceder a un elemento mediante su posición. En una tabla escribimos `array[fila, columna]`, recordando que los índices empiezan en cero. Así, `temperaturas[1, 2]` selecciona la segunda estación y el tercer día.

Un **corte** o *slice* selecciona un intervalo. La notación `inicio:fin` incluye el índice inicial y excluye el final. Por ejemplo, `temperaturas[:, 1:3]` selecciona todas las filas y las columnas con índices uno y dos. Los índices negativos cuentan desde el final, de modo que `-2:` recoge los dos últimos elementos de un eje.

También podemos seleccionar por una condición. La comparación `temperaturas > 20` genera una **máscara booleana** con la misma forma que los datos, formada por valores `True` y `False`. Al escribir `temperaturas[mascara_calor]` obtenemos los valores que cumplen la condición en un array de una dimensión.

<div style="padding: 0.8em 1em; margin: 0.8em 0 1em; border-left: 4px solid #C58A32; background-color: rgba(197,138,50,0.08); border-radius: 4px; line-height: 1.6;">
<p style="margin: 0.45em 0;">Un corte de un array, como <code>temperaturas[:, 1:3]</code>, es una <strong>vista</strong> que comparte los datos con el original. Modificarla puede cambiar también el array de partida. Para trabajar de forma independiente utilizamos <code>.copy()</code>. En cambio, la selección mediante una máscara booleana devuelve una copia de los valores seleccionados.</p>
</div>
<p style="font-size: 0.9em; margin: 0.8em 0 0.3em;"><a href="https://numpy.org/doc/stable/user/basics.copies.html">Copias y vistas en NumPy</a></p>

In [49]:
# Tres estaciones y cuatro días de temperaturas sintéticas, en °C.
temperaturas = np.array([
    [18.0, 19.2, 20.1, 18.9],  # Estación costera
    [16.5, 17.0, 17.8, 17.3],  # Estación de valle
    [21.2, 21.5, 22.0, 21.7],  # Estación urbana
])

estacion_2_dia_3 = temperaturas[1, 2]
print("Temperatura de la estación 2 en el día 3:", estacion_2_dia_3, "°C")

# Los dos cortes conservan todas las estaciones.
submatriz = temperaturas[:, 1:3]
ultimos_dias = temperaturas[:, -2:]
print("\nDías 2 y 3 (°C):\n", submatriz)
print("Últimos dos días (°C):\n", ultimos_dias)

mascara_calor = temperaturas > 20
valores_calor = temperaturas[mascara_calor]
print("\nMáscara de temperaturas superiores a 20 °C:\n", mascara_calor)
print("Valores seleccionados (°C):", valores_calor)
print("Forma de la selección:", valores_calor.shape)

# Modificamos una copia del corte para conservar intactos los datos originales.
submatriz_copia = submatriz.copy()
submatriz_copia[0, 0] += 1.0
print("\nPrimer valor del corte original:", submatriz[0, 0], "°C")
print("Primer valor de la copia modificada:", submatriz_copia[0, 0], "°C")

Temperatura de la estación 2 en el día 3: 17.8 °C

Días 2 y 3 (°C):
 [[19.2 20.1]
 [17.  17.8]
 [21.5 22. ]]
Últimos dos días (°C):
 [[20.1 18.9]
 [17.8 17.3]
 [22.  21.7]]

Máscara de temperaturas superiores a 20 °C:
 [[False False  True False]
 [False False False False]
 [ True  True  True  True]]
Valores seleccionados (°C): [20.1 21.2 21.5 22.  21.7]
Forma de la selección: (5,)

Primer valor del corte original: 19.2 °C
Primer valor de la copia modificada: 20.2 °C


### 2.2 Broadcasting para combinar patrones

<div style="padding: 0.8em 1em; margin: 0.8em 0 1em; border-left: 4px solid #4C78A8; background-color: rgba(76,120,168,0.08); border-radius: 4px; line-height: 1.6;">
<p style="margin: 0.45em 0;">El <strong>broadcasting</strong> permite combinar arrays de formas diferentes cuando sus ejes son compatibles. Podemos imaginar que el array más pequeño se repite hasta encajar con el otro, aunque NumPy no necesita almacenar esas repeticiones para realizar la operación.</p>
</div>

El caso más sencillo consiste en sumar una corrección escalar a una serie. En `serie_temperatura + 0.3`, el mismo valor se suma a cada elemento. También podemos combinar una tabla de cinco años y doce meses, de forma `(5, 12)`, con una referencia mensual de forma `(12,)`. Cada mes recibe el valor de referencia que le corresponde.

**Reglas de compatibilidad**

1. Se comparan las formas desde el último eje hacia la izquierda.
2. En cada posición, los tamaños deben ser iguales o uno de ellos debe ser uno.
3. Los ejes que faltan se interpretan como ejes de tamaño uno.
4. Cuando un tamaño uno se combina con otro tamaño, el resultado adopta este último. Si los tamaños ya coinciden, se conservan.

<div style="overflow-x: auto; margin: 0.8em 0 1.2em;">
<table style="border-collapse: collapse; width: 100%; font-size: 0.96em; line-height: 1.5; text-align: left; table-layout: fixed;">
<colgroup><col style="width: 28%;"><col style="width: 29%;"><col style="width: 43%;"></colgroup>
<thead><tr style="background-color: rgba(76,120,168,0.10);">
<th scope="col" style="padding: 0.6em 0.8em; text-align: left; border-bottom: 2px solid rgba(114,160,193,0.55);">Formas de entrada</th>
<th scope="col" style="padding: 0.6em 0.8em; text-align: left; border-bottom: 2px solid rgba(114,160,193,0.55);">Cómo se comparan</th>
<th scope="col" style="padding: 0.6em 0.8em; text-align: left; border-bottom: 2px solid rgba(114,160,193,0.55);">Resultado</th>
</tr></thead>
<tbody>
<tr>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;"><code>(3,)</code> y <code>(2, 1)</code></td>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;"><code>(1, 3)</code> y <code>(2, 1)</code></td>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;"><code>(2, 3)</code>. Cada eje puede ampliarse donde aparece un uno.</td>
</tr>
<tr>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;"><code>(5, 12)</code> y <code>(12,)</code></td>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;"><code>(5, 12)</code> y <code>(1, 12)</code></td>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;"><code>(5, 12)</code>. La referencia mensual se aplica a los cinco años.</td>
</tr>
<tr>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;"><code>(2, 3)</code> y <code>(4,)</code></td>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;"><code>(2, 3)</code> y <code>(1, 4)</code></td>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;">Incompatibles. Los tamaños tres y cuatro del último eje no coinciden y ninguno es uno.</td>
</tr>
</tbody></table>
</div>

Cuando las formas no son compatibles, NumPy genera un `ValueError`. La posición de los ejes importa tanto como su tamaño. Una forma `(2,)` no indica por sí sola si los dos valores corresponden a estaciones, días o cualquier otra variable.

**Qué ocurre en memoria**

Los **strides** indican cuántos bytes hay que avanzar en memoria al desplazarse un elemento por cada eje. Junto con la forma y el tipo de dato, describen cómo se accede a los valores almacenados. Durante el broadcasting, un eje puede recorrerse con un desplazamiento de cero bytes, reutilizando el mismo dato en lugar de copiarlo repetidamente.

<div style="padding: 0.8em 1em; margin: 0.8em 0 1em; border-left: 3px solid #72A0C1; background-color: rgba(76,120,168,0.025); border-radius: 4px; line-height: 1.6;">
<p style="margin: 0.45em 0;">El broadcasting evita duplicar las entradas solo para igualar sus formas, pero el resultado de una operación puede ocupar memoria adicional. Combinar arrays pequeños puede producir un resultado grande. Para utilizarlo basta con comprender las reglas de compatibilidad; los <em>strides</em> explican cómo se hace posible internamente.</p>
</div>

En el ejemplo aplicaremos primero una corrección escalar y después combinaremos una referencia mensual sintética con anomalías de cinco años. La relación utilizada es $T = C + A$, donde $T$ es la temperatura, $C$ la referencia y $A$ la diferencia respecto de ella, expresadas aquí en grados Celsius.
<p style="font-size: 0.9em; margin: 0.8em 0 0.3em;"><a href="https://numpy.org/doc/stable/user/basics.broadcasting.html">Reglas de broadcasting</a> · <a href="https://numpy.org/doc/stable/reference/generated/numpy.ndarray.strides.html">Strides de un array</a></p>

In [50]:
# La misma corrección se suma a las siete temperaturas, en °C.
serie_temperatura = np.array([20.1, 19.6, 18.9, 19.2, 20.4, 21.0, 20.3])
correccion_equipos = 0.3
serie_corregida = serie_temperatura + correccion_equipos
print("Serie original (°C):", serie_temperatura)
print("Serie corregida (°C):", serie_corregida)

# Referencia mensual inventada, de enero a diciembre, en °C.
# Representa una climatología para practicar; no procede de observaciones.
climatologia_mensual = np.array([
    10.0, 11.0, 12.5, 14.5, 18.0, 22.0,
    25.0, 25.2, 22.0, 18.0, 13.5, 11.0,
])
anomalia_base = np.array([
    0.3, 0.2, 0.1, 0.1, 0.4, 0.8, 1.1, 1.0, 0.5, 0.3, 0.1, 0.2,
])

# Una fila por año y una columna para extender el ajuste a sus doce meses.
desplazamiento_por_ano = np.array([[0.0], [0.1], [0.2], [0.3], [0.4]])
anomalias_mensuales = anomalia_base + desplazamiento_por_ano  # (5, 12)
temperatura_modelo = anomalias_mensuales + climatologia_mensual

print("\nForma de la referencia mensual:", climatologia_mensual.shape)
print("Forma de las anomalías:", anomalias_mensuales.shape)
print("Temperaturas del primer año (°C):", np.round(temperatura_modelo[0], 1))
print("Temperaturas de los cinco años (°C):\n", np.round(temperatura_modelo, 1))
print("Forma del resultado:", temperatura_modelo.shape)

Serie original (°C): [20.1 19.6 18.9 19.2 20.4 21.  20.3]
Serie corregida (°C): [20.4 19.9 19.2 19.5 20.7 21.3 20.6]

Forma de la referencia mensual: (12,)
Forma de las anomalías: (5, 12)
Temperaturas del primer año (°C): [10.3 11.2 12.6 14.6 18.4 22.8 26.1 26.2 22.5 18.3 13.6 11.2]
Temperaturas de los cinco años (°C):
 [[10.3 11.2 12.6 14.6 18.4 22.8 26.1 26.2 22.5 18.3 13.6 11.2]
 [10.4 11.3 12.7 14.7 18.5 22.9 26.2 26.3 22.6 18.4 13.7 11.3]
 [10.5 11.4 12.8 14.8 18.6 23.  26.3 26.4 22.7 18.5 13.8 11.4]
 [10.6 11.5 12.9 14.9 18.7 23.1 26.4 26.5 22.8 18.6 13.9 11.5]
 [10.7 11.6 13.  15.  18.8 23.2 26.5 26.6 22.9 18.7 14.  11.6]]
Forma del resultado: (5, 12)


Podemos añadir un eje de tamaño uno con `np.newaxis` o reorganizar la forma con `reshape()` para indicar cómo deben alinearse los datos. Esto no cambia los valores ni convierte en compatibles dos tamaños arbitrarios.

Si tenemos una tabla de dos estaciones y siete días, su forma es `(2, 7)`. Una corrección por estación de forma `(2,)` no puede sumarse directamente, porque NumPy compara ese dos con el siete del último eje. Al convertirla en `(2, 1)`, cada corrección se aplica a los siete días de su estación.

<div style="padding: 0.8em 1em; margin: 0.8em 0 1em; border-left: 4px solid #59A14F; background-color: rgba(89,161,79,0.08); border-radius: 4px; line-height: 1.6;">
<p style="margin: 0.45em 0;">Un array de forma <code>(2,)</code> tiene un eje; uno de forma <code>(2, 1)</code> tiene dos. Aunque ambos contienen dos valores, expresan una organización diferente. Antes de operar, comprueba qué representa cada eje, incluso cuando NumPy acepta las formas.</p>
</div>

El ejemplo también muestra que una referencia de forma `(12,)` puede escribirse como `(1, 12)` mediante `np.newaxis`. En ese caso el nuevo eje no es imprescindible para sumarla a una tabla `(2, 12)`, pero ayuda a hacer explícita la correspondencia.

In [51]:
# Dos estaciones y siete días de temperatura media, en °C.
temperaturas_estaciones = np.array([
    [20.1, 19.6, 18.9, 19.2, 20.4, 21.0, 20.3],
    [15.4, 14.8, 14.2, 14.0, 15.0, 15.6, 15.3],
])
correccion_estaciones = np.array([0.3, 0.5])  # Ajuste que se suma, en °C.

# Dos valores pasan a organizarse en dos filas y una columna.
correccion_2x1 = correccion_estaciones.reshape(2, 1)
correccion_con_eje = correccion_estaciones[:, np.newaxis]  # Misma forma.
print("Forma de las temperaturas:", temperaturas_estaciones.shape)
print("Forma de las correcciones originales:", correccion_estaciones.shape)
print("Forma tras reshape:", correccion_2x1.shape)
print("Forma con np.newaxis:", correccion_con_eje.shape)

# (2, 7) + (2, 1) aplica una corrección distinta a cada fila.
series_corregidas = temperaturas_estaciones + correccion_2x1
print("\nTemperaturas originales (°C):\n", temperaturas_estaciones)
print("Temperaturas corregidas (°C):\n", series_corregidas)

# Dos estaciones ficticias comparten la referencia mensual del ejemplo anterior.
# Esta simplificación sirve para practicar la operación, no para comparar climas.
anomalias_por_estacion = np.array([
    [0.2, 0.1, 0.0, 0.1, 0.4, 0.7, 1.0, 0.9, 0.5, 0.2, 0.1, 0.1],
    [-0.5, -0.4, -0.3, 0.0, 0.3, 0.9, 1.3, 1.1, 0.7, 0.1, -0.2, -0.4],
])
clima_1x12 = climatologia_mensual[np.newaxis, :]  # (12,) pasa a (1, 12).
temperatura_media = clima_1x12 + anomalias_por_estacion

print("\nForma de la referencia con eje añadido:", clima_1x12.shape)
print("Forma de las anomalías por estación:", anomalias_por_estacion.shape)
print("Temperaturas mensuales por estación (°C):\n", np.round(temperatura_media, 1))
print("Forma del resultado:", temperatura_media.shape)

Forma de las temperaturas: (2, 7)
Forma de las correcciones originales: (2,)
Forma tras reshape: (2, 1)
Forma con np.newaxis: (2, 1)

Temperaturas originales (°C):
 [[20.1 19.6 18.9 19.2 20.4 21.  20.3]
 [15.4 14.8 14.2 14.  15.  15.6 15.3]]
Temperaturas corregidas (°C):
 [[20.4 19.9 19.2 19.5 20.7 21.3 20.6]
 [15.9 15.3 14.7 14.5 15.5 16.1 15.8]]

Forma de la referencia con eje añadido: (1, 12)
Forma de las anomalías por estación: (2, 12)
Temperaturas mensuales por estación (°C):
 [[10.2 11.1 12.5 14.6 18.4 22.7 26.  26.1 22.5 18.2 13.6 11.1]
 [ 9.5 10.6 12.2 14.5 18.3 22.9 26.3 26.3 22.7 18.1 13.3 10.6]]
Forma del resultado: (2, 12)


### 2.3 Funciones universales (ufuncs) y cambio de unidades

Las **funciones universales** o **ufuncs** aplican una operación elemento a elemento. Entre ellas se encuentran `np.multiply()`, `np.sin()` y `np.cos()`. Admiten broadcasting y utilizan rutinas compiladas para los tipos numéricos habituales.

Para convertir una velocidad de metros por segundo a kilómetros por hora multiplicamos sus valores por 3,6. En un array, `velocidad_viento * 3.6` y `np.multiply(velocidad_viento, 3.6)` expresan la misma operación.

$$
V_{\mathrm{km/h}} = 3.6\,V_{\mathrm{m/s}}.
$$

**Ángulos y componentes del viento**

Las funciones trigonométricas de NumPy trabajan en radianes. Si los ángulos están en grados, `np.deg2rad()` realiza la conversión.

En la convención matemática, un ángulo $\theta$ medido desde el este y en sentido antihorario indica hacia dónde apunta el vector. Sus componentes se calculan mediante

$$
u = V\cos(\theta), \qquad v = V\sin(\theta).
$$

La dirección meteorológica $d$ indica, en cambio, **de dónde viene el viento**, medida desde el norte y en sentido horario. Con $d$ en radianes, las componentes son

$$
u = -V\sin(d), \qquad v = -V\cos(d).
$$

En ambas expresiones, $u$ es positiva hacia el este y $v$ hacia el norte. Si la rapidez $V$ se expresa en $\mathrm{m\,s^{-1}}$, las componentes conservan esas unidades. Un viento del norte tiene $u=0$ y $v<0$.

<div style="padding: 0.8em 1em; margin: 0.8em 0 1em; border-left: 4px solid #59A14F; background-color: rgba(89,161,79,0.08); border-radius: 4px; line-height: 1.6;">
<p style="margin: 0.45em 0;">En general, un mismo valor angular representa vientos distintos según la convención. El ejemplo las muestra por separado para comparar las fórmulas. En datos reales debemos conocer la convención antes de calcular las componentes.</p>
</div>

**Selección y limitación de valores**

Otras herramientas complementan estas operaciones. `np.where(condicion, valor_si, valor_no)` elige un resultado en cada posición según una condición. `np.clip()` limita los valores a un intervalo; no es equivalente a identificar o corregir observaciones erróneas.

<div style="padding: 0.8em 1em; margin: 0.8em 0 1em; border-left: 4px solid #C58A32; background-color: rgba(197,138,50,0.08); border-radius: 4px; line-height: 1.6;">
<p style="margin: 0.45em 0;">El umbral de 7 m/s utilizado aquí es arbitrario y solo ilustra la sintaxis. Recortar todas las velocidades superiores a ese valor modifica los datos y elimina extremos que podrían ser válidos. Por eso conservamos el array original y no presentamos el recorte como un control de calidad meteorológico.</p>
</div>
<p style="font-size: 0.9em; margin: 0.8em 0 0.3em;"><a href="https://numpy.org/doc/stable/user/basics.ufuncs.html">Funciones universales</a> · <a href="https://confluence.ecmwf.int/spaces/CKB/pages/133262398/ERA5+How+to+calculate+wind+speed+and+wind+direction+from+u+and+v+components+of+the+wind">Componentes y dirección del viento — ECMWF</a> · <a href="https://numpy.org/doc/stable/reference/generated/numpy.clip.html">Limitación con clip</a></p>

In [52]:
# Velocidades sintéticas de cinco días consecutivos, en m/s.
velocidad_viento = np.array([4.5, 6.2, 8.0, 5.1, 7.3])
velocidad_kmh = np.multiply(velocidad_viento, 3.6)
print("Velocidades en km/h:", np.round(velocidad_kmh, 2))

# Ángulos matemáticos que indican hacia dónde apunta el vector.
angulos_matematicos_grados = np.array([0, 45, 90, 135, 180])
angulos_matematicos_rad = np.deg2rad(angulos_matematicos_grados)
componente_u = velocidad_viento * np.cos(angulos_matematicos_rad)
componente_v = velocidad_viento * np.sin(angulos_matematicos_rad)
print("\nConvención matemática — u (m/s):", np.round(componente_u, 2))
print("Convención matemática — v (m/s):", np.round(componente_v, 2))

# Ahora interpretamos esos números como direcciones meteorológicas.
# No convertimos las direcciones anteriores; ahora 0° indica procedencia del norte.
direcciones_meteo_grados = np.array([0, 45, 90, 135, 180])
direcciones_meteo_rad = np.deg2rad(direcciones_meteo_grados)
u_met = -velocidad_viento * np.sin(direcciones_meteo_rad)
v_met = -velocidad_viento * np.cos(direcciones_meteo_rad)
print("\nConvención meteorológica — u (m/s):", np.round(u_met, 2))
print("Convención meteorológica — v (m/s):", np.round(v_met, 2))

# Clasificación con un umbral elegido solo para el ejemplo.
umbral_viento = 7.0  # m/s
mascara_viento = velocidad_viento > umbral_viento
etiqueta_viento = np.where(mascara_viento, "supera el umbral", "no lo supera")
print("\nMáscara V > 7 m/s:", mascara_viento)
print("Etiquetas:", etiqueta_viento)

# El recorte devuelve otro array; no sustituye ni valida las observaciones.
velocidad_limitada = np.clip(velocidad_viento, a_min=None, a_max=umbral_viento)
print("\nVelocidades originales (m/s):", velocidad_viento)
print("Velocidades limitadas (m/s):", velocidad_limitada)

# Recalculamos las componentes matemáticas de las velocidades limitadas.
u_limitada = velocidad_limitada * np.cos(angulos_matematicos_rad)
v_limitada = velocidad_limitada * np.sin(angulos_matematicos_rad)
print("Componentes u tras el recorte (m/s):", np.round(u_limitada, 2))
print("Componentes v tras el recorte (m/s):", np.round(v_limitada, 2))

Velocidades en km/h: [16.2  22.32 28.8  18.36 26.28]

Convención matemática — u (m/s): [ 4.5   4.38  0.   -3.61 -7.3 ]
Convención matemática — v (m/s): [0.   4.38 8.   3.61 0.  ]

Convención meteorológica — u (m/s): [-0.   -4.38 -8.   -3.61 -0.  ]
Convención meteorológica — v (m/s): [-4.5  -4.38 -0.    3.61  7.3 ]

Máscara V > 7 m/s: [False False  True False  True]
Etiquetas: ['no lo supera' 'no lo supera' 'supera el umbral' 'no lo supera'
 'supera el umbral']

Velocidades originales (m/s): [4.5 6.2 8.  5.1 7.3]
Velocidades limitadas (m/s): [4.5 6.2 7.  5.1 7. ]
Componentes u tras el recorte (m/s): [ 4.5   4.38  0.   -3.61 -7.  ]
Componentes v tras el recorte (m/s): [0.   4.38 7.   3.61 0.  ]


## 3. Estadística descriptiva aplicada a climatología

La estadística descriptiva permite resumir un conjunto de observaciones mediante medidas como la media, la mediana o la desviación estándar. Antes de calcularlas, debemos identificar qué representa cada dato, qué periodo cubre la serie y cómo están indicados los valores ausentes.

Cuando los datos tienen varios ejes, el argumento **`axis`** indica cuál se recorre para calcular el estadístico. En una tabla de estaciones y días, la interpretación es la siguiente.

<div style="overflow-x: auto; margin: 0.8em 0 1.2em;">
<table style="border-collapse: collapse; width: 100%; font-size: 0.96em; line-height: 1.5; text-align: left; table-layout: fixed;">
<colgroup><col style="width: 23%;"><col style="width: 45%;"><col style="width: 32%;"></colgroup>
<thead><tr style="background-color: rgba(76,120,168,0.10);">
<th scope="col" style="padding: 0.6em 0.8em; text-align: left; border-bottom: 2px solid rgba(114,160,193,0.55);">Argumento</th>
<th scope="col" style="padding: 0.6em 0.8em; text-align: left; border-bottom: 2px solid rgba(114,160,193,0.55);">Qué se resume</th>
<th scope="col" style="padding: 0.6em 0.8em; text-align: left; border-bottom: 2px solid rgba(114,160,193,0.55);">Qué se obtiene</th>
</tr></thead>
<tbody>
<tr>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;"><code>axis=None</code></td>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;">Todos los elementos.</td>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;">Un único valor para el conjunto.</td>
</tr>
<tr>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;"><code>axis=0</code></td>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;">Las estaciones, manteniendo separado cada día.</td>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;">Un valor por día.</td>
</tr>
<tr>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;"><code>axis=1</code></td>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;">Los días, manteniendo separada cada estación.</td>
<td style="padding: 0.55em 0.8em; text-align: left; vertical-align: top; border-bottom: 1px solid rgba(114,160,193,0.28); overflow-wrap: anywhere;">Un valor por estación.</td>
</tr>
</tbody></table>
</div>

### 3.1 Datos faltantes y estadísticos básicos

Trabajaremos con doce acumulados mensuales de precipitación, en milímetros, de los que faltan dos. Utilizamos `np.nan` para representar esos datos ausentes. La función `np.isnan()` permite localizarlos; compararlos mediante `== np.nan` no sirve para detectarlos.

Las funciones `np.nanmean()`, `np.nanmedian()` y `np.nanstd()` excluyen los `NaN` al calcular el estadístico, pero **no rellenan los huecos ni modifican la serie**. En este ejemplo la media representa el promedio de los meses disponibles, no una precipitación anual ni una climatología de cada mes.

<div style="padding: 0.8em 1em; margin: 0.8em 0 1em; border-left: 4px solid #C58A32; background-color: rgba(197,138,50,0.08); border-radius: 4px; line-height: 1.6;">
<p style="margin: 0.45em 0;">Conviene acompañar el estadístico del número de datos válidos. Si todos los valores son <code>NaN</code>, no hay observaciones con las que calcular la media o la desviación. Estas funciones devuelven <code>NaN</code> y pueden emitir una advertencia. Omitir datos ausentes tampoco elimina el posible sesgo debido a su ausencia.</p>
</div>

Por defecto, `np.nanstd()` utiliza `ddof=0`, que divide la suma de desviaciones cuadráticas por el número de valores válidos antes de tomar la raíz cuadrada. Con `ddof=1` utiliza un dato menos en el denominador, una corrección habitual en la estimación muestral de la varianza. Aquí mantendremos `ddof=0` para describir la serie disponible.

Finalmente, restaremos la media de los meses observados a cada dato. Estas diferencias son **desviaciones respecto de la media de la muestra**. Para obtener anomalías climatológicas mensuales necesitaríamos una referencia apropiada para cada mes del calendario.
<p style="font-size: 0.9em; margin: 0.8em 0 0.3em;"><a href="https://numpy.org/doc/stable/reference/generated/numpy.nanstd.html">Desviación estándar con datos ausentes</a> · <a href="https://wmo.int/wmo-climatological-normals">Referencias climatológicas — OMM</a></p>

In [53]:
# Acumulados mensuales sintéticos, de enero a diciembre, en mm.
precipitacion_mm = np.array([
    45.2, 38.0, np.nan, 22.5, 15.0, 10.1,
    5.3, np.nan, 30.2, 48.7, 52.4, 60.0,
])

datos_faltantes = np.isnan(precipitacion_mm)
numero_validos = np.count_nonzero(~datos_faltantes)  # ~ invierte la máscara.
print("Máscara de meses sin dato:", datos_faltantes)
print("Número de meses disponibles:", numero_validos, "de", precipitacion_mm.size)

media_prec = np.nanmean(precipitacion_mm)
mediana_prec = np.nanmedian(precipitacion_mm)
desviacion_prec = np.nanstd(precipitacion_mm, ddof=0)
print(f"\nMedia de los meses disponibles: {media_prec:.1f} mm")
print(f"Mediana: {mediana_prec:.1f} mm")
print(f"Desviación estándar (ddof=0): {desviacion_prec:.1f} mm")

# Restamos la misma referencia a todos los meses; los NaN se conservan.
# No son anomalías respecto de una climatología mensual de varios años.
desviaciones_prec = precipitacion_mm - media_prec
print("\nDesviaciones respecto de la media de la muestra (mm):")
print(np.round(desviaciones_prec, 1))

Máscara de meses sin dato: [False False  True False False False False  True False False False False]
Número de meses disponibles: 10 de 12

Media de los meses disponibles: 32.7 mm
Mediana: 34.1 mm
Desviación estándar (ddof=0): 18.1 mm

Desviaciones respecto de la media de la muestra (mm):
[ 12.5   5.3   nan -10.2 -17.7 -22.6 -27.4   nan  -2.5  16.   19.7  27.3]


### 3.2 Resumen térmico y anomalías diarias

Generaremos treinta temperaturas máximas diarias sintéticas para practicar el cálculo de una media, una desviación estándar y las diferencias de cada día respecto de la media del periodo. También identificaremos los valores que superan un umbral ilustrativo mediante una máscara booleana.

El umbral será $\overline{T} + 1.5\,\sigma$, donde $\overline{T}$ es la media de las treinta máximas y $\sigma$ su desviación estándar con `ddof=0`. Ambas se calculan a partir de esta misma muestra y se expresan en grados Celsius.

<div style="padding: 0.8em 1em; margin: 0.8em 0 1em; border-left: 4px solid #C58A32; background-color: rgba(197,138,50,0.08); border-radius: 4px; line-height: 1.6;">
<p style="margin: 0.45em 0;">La media de un solo mes no constituye una climatología. Del mismo modo, seleccionar días por encima de este umbral no identifica por sí solo una <strong>ola de calor</strong>. Para ello harían falta una referencia climática y criterios de persistencia y, según la definición, extensión espacial. Aquí estudiamos únicamente superaciones de un umbral de ejemplo.</p>
</div>

Los valores se generan a partir de una distribución normal para disponer de una muestra reproducible. No representan observaciones ni un modelo de la evolución temporal de la temperatura.
<p style="font-size: 0.9em; margin: 0.8em 0 0.3em;"><a href="https://wmo.int/wmo-climatological-normals">Referencia climática — OMM</a> · <a href="https://www.aemet.es/es/conocermas/recursos_en_linea/publicaciones_y_estudios/estudios/detalles/olascalor">Criterios de ola de calor — AEMET</a></p>

In [54]:
# La semilla permite repetir esta muestra sin alterar otros generadores.
rng_temperaturas = np.random.default_rng(7)
maximas_diarias = rng_temperaturas.normal(loc=29, scale=4, size=30)  # °C

media_mensual = maximas_diarias.mean()
desviacion_mensual = maximas_diarias.std(ddof=0)
anomalias_diarias = maximas_diarias - media_mensual
print(f"Media de las máximas del mes simulado: {media_mensual:.2f} °C")
print(f"Desviación estándar: {desviacion_mensual:.2f} °C")
print("Diferencias respecto de esa media (°C):", np.round(anomalias_diarias, 2))

# Criterio didáctico de superación, no una definición de ola de calor.
umbral_temperatura = media_mensual + 1.5 * desviacion_mensual
supera_umbral = maximas_diarias > umbral_temperatura
print(f"\nUmbral del ejemplo: {umbral_temperatura:.2f} °C")
print("Máscara de superación:", supera_umbral)
print("Número de días por encima del umbral:", np.count_nonzero(supera_umbral))

# La misma máscara selecciona las temperaturas y sus días correspondientes.
dias_del_mes = np.arange(1, maximas_diarias.size + 1)
print("Días seleccionados:", dias_del_mes[supera_umbral])
print("Temperaturas seleccionadas (°C):", np.round(maximas_diarias[supera_umbral], 2))

Media de las máximas del mes simulado: 27.34 °C
Desviación estándar: 3.29 °C
Diferencias respecto de esa media (°C): [ 1.66  2.85  0.56 -1.91 -0.16 -2.31  1.9   7.02 -0.31 -0.83  3.62  3.08
  2.08 -2.07  1.54  4.44 -3.72 -0.17 -5.95 -3.5  -5.71  0.72 -3.41  2.74
  2.28  0.91 -8.41 -0.5   1.46  2.11]

Umbral del ejemplo: 32.28 °C
Máscara de superación: [False False False False False False False  True False False False False
 False False False False False False False False False False False False
 False False False False False False]
Número de días por encima del umbral: 1
Días seleccionados: [8]
Temperaturas seleccionadas (°C): [34.36]


## 4. Reorganización de dimensiones y operaciones vectorizadas

Los campos meteorológicos pueden organizarse, por ejemplo, como `(tiempo, latitud, longitud)`. A veces resulta útil reunir las dos dimensiones espaciales en una sola para trabajar con una tabla de tiempos y puntos. Esta operación reorganiza los datos, pero no interpola ni cambia la resolución de la malla.

### 4.1 Cambio temporal de forma con reshape

El método `reshape()` devuelve un array con la forma solicitada, siempre que se conserve el número de elementos. Podemos usar `-1` en una única posición para que NumPy deduzca ese tamaño. Así, una forma `(3, 2, 4)` puede transformarse en `(3, 8)` mediante `reshape(3, -1)`.

<div style="padding: 0.8em 1em; margin: 0.8em 0 1em; border-left: 3px solid #72A0C1; background-color: rgba(76,120,168,0.025); border-radius: 4px; line-height: 1.6;">
<p style="margin: 0.45em 0;"><code>reshape()</code> devuelve una vista cuando la organización en memoria lo permite y, en caso contrario, puede copiar los datos. No debe suponerse que siempre evita las copias. Tampoco equivale a intercambiar ejes, una operación para la que existen herramientas como <code>transpose()</code>.</p>
</div>

Conservaremos la forma original en `t, ny, nx` para poder reconstruirla después. Con el orden predeterminado, la longitud es el índice espacial que cambia más rápidamente al recorrer cada fila de la tabla resultante.

**Coordenadas de la malla**

`np.meshgrid()` permite construir coordenadas de dos dimensiones a partir de vectores de latitud y longitud. Con `indexing="ij"`, el primer eje corresponde a las latitudes y el segundo a las longitudes. Cada par `lat2d[i, j]`, `lon2d[i, j]` identifica la posición del dato `temperatura_3d[k, i, j]` en el instante `k`.

En el ejemplo calcularemos una media de los ocho puntos para cada instante y recuperaremos la forma original. Esa media da el mismo peso a todos los puntos. Una media espacial sobre una malla real puede requerir ponderar por el área representada por cada celda.

<div style="padding: 0.8em 1em; margin: 0.8em 0 1em; border-left: 4px solid #59A14F; background-color: rgba(89,161,79,0.08); border-radius: 4px; line-height: 1.6;">
<p style="margin: 0.45em 0;">Reorganizar y recuperar la forma no realiza cálculos sobre los valores, por lo que podemos comprobar una igualdad exacta con <code>np.array_equal()</code>. Tras operaciones de coma flotante, <code>np.allclose()</code> permite comparar con una tolerancia que conviene elegir según la precisión y las unidades del problema.</p>
</div>
<p style="font-size: 0.9em; margin: 0.8em 0 0.3em;"><a href="https://numpy.org/doc/stable/reference/generated/numpy.reshape.html">Reorganización con reshape</a> · <a href="https://numpy.org/doc/stable/reference/generated/numpy.meshgrid.html">Coordenadas con meshgrid</a></p>

In [55]:
# Campo sintético en kelvin, organizado como (tiempo, latitud, longitud).
rng_campo = np.random.default_rng(0)
temperatura_3d = rng_campo.uniform(low=270, high=310, size=(3, 2, 4))
t, ny, nx = temperatura_3d.shape
print("Forma original:", temperatura_3d.shape)

latitudes_malla = np.array([-45.0, 45.0])
longitudes_malla = np.array([0.0, 90.0, 180.0, 270.0])
lat2d, lon2d = np.meshgrid(latitudes_malla, longitudes_malla, indexing="ij")
print("\nLatitudes de la malla (grados):\n", lat2d)
print("Longitudes de la malla (grados):\n", lon2d)

# Se conserva cada instante y se reúnen sus 2 × 4 puntos espaciales.
tabla_tiempo_puntos = temperatura_3d.reshape(t, -1)
print("\nForma reorganizada (tiempo, puntos):", tabla_tiempo_puntos.shape)

# axis=1 resume los puntos de cada fila, no los instantes.
media_por_tiempo = tabla_tiempo_puntos.mean(axis=1)
print("Media no ponderada por instante (K):", np.round(media_por_tiempo, 2))

recuperado = tabla_tiempo_puntos.reshape(t, ny, nx)
print("Forma recuperada:", recuperado.shape)
print("Coincidencia exacta con el original:", np.array_equal(temperatura_3d, recuperado))
assert np.array_equal(temperatura_3d, recuperado)

# También podemos reunir todos los ejes o resumir varios a la vez.
serie_aplanada = temperatura_3d.ravel()
media_conjunto = temperatura_3d.mean()
media_por_latitud = temperatura_3d.mean(axis=(0, 2))
print("\nForma tras aplanar todos los ejes:", serie_aplanada.shape)
print(f"Media de todos los valores: {media_conjunto:.2f} K")
print("Media por latitud, sobre tiempos y longitudes (K):", np.round(media_por_latitud, 2))

Forma original: (3, 2, 4)

Latitudes de la malla (grados):
 [[-45. -45. -45. -45.]
 [ 45.  45.  45.  45.]]
Longitudes de la malla (grados):
 [[  0.  90. 180. 270.]
 [  0.  90. 180. 270.]]

Forma reorganizada (tiempo, puntos): (3, 8)
Media no ponderada por instante (K): [290.13 290.47 287.99]
Forma recuperada: (3, 2, 4)
Coincidencia exacta con el original: True

Forma tras aplanar todos los ejes: (24,)
Media de todos los valores: 289.53 K
Media por latitud, sobre tiempos y longitudes (K): [287.96 291.1 ]


# Ejercicios para practicar

<div style="padding: 0.8em 1em; margin: 0.8em 0 1em; border-left: 4px solid #4C78A8; background-color: rgba(76,120,168,0.08); border-radius: 4px; line-height: 1.6;">
<p style="margin: 0.45em 0;"><strong>1. Medias y recuentos básicos</strong><br>Utiliza el array <code>temperaturas_diarias</code> para calcular la media, la mediana y la desviación estándar con <code>ddof=0</code>. Expresa los resultados en grados Celsius y determina cuántos días registraron temperaturas estrictamente superiores a <strong>21 °C</strong>.</p>
</div>

In [56]:
print(temperaturas_diarias)
media_temp = np.mean(temperaturas_diarias)
mediana_temp = np.median(temperaturas_diarias)
desviacion_temp = temperaturas_diarias.std(ddof=0)
print(f"Media: {media_temp:.2f}")
print(f"Mediana: {mediana_temp: .2f}")
print(f"Desviación: {desviacion_temp: .2f}")

[18.2 19.7 20.1 19.5 21.3 22.  20.8]
Media: 20.23
Mediana:  20.10
Desviación:  1.17


<div style="padding: 0.8em 1em; margin: 0.8em 0 1em; border-left: 4px solid #4C78A8; background-color: rgba(76,120,168,0.08); border-radius: 4px; line-height: 1.6;">
<p style="margin: 0.45em 0;"><strong>2. Rejilla de estaciones y días</strong><br>A partir de <code>temperaturas_rejilla</code>, calcula la temperatura media de cada estación con <code>axis=1</code> y la de cada día con <code>axis=0</code>. Comprueba cuántos valores devuelve cada operación y cómo se relacionan con la forma del array.<br><br>Después crea una copia de la tabla. Mediante una máscara booleana, conserva las temperaturas estrictamente superiores a <strong>19 °C</strong> y sustituye las restantes por <code>np.nan</code>, manteniendo la forma de la tabla y los datos originales.</p>
</div>

In [57]:
print(temperaturas_rejilla)
media_estacion = temperaturas_rejilla.mean(axis=1)
media_dia = temperaturas_rejilla.mean(axis=0)

print("Media estación:", media_estacion)
print("Media dia:", media_dia)

[[17.5 18.1 19.  18.6]
 [15.2 15.8 16.4 16. ]
 [20.1 20.5 21.2 20.9]]
Media estación: [18.3   15.85  20.675]
Media dia: [17.6        18.13333333 18.86666667 18.5       ]


<div style="padding: 0.8em 1em; margin: 0.8em 0 1em; border-left: 4px solid #4C78A8; background-color: rgba(76,120,168,0.08); border-radius: 4px; line-height: 1.6;">
<p style="margin: 0.45em 0;"><strong>3. Serie equiespaciada de presión</strong><br>Con <code>np.linspace()</code>, construye una serie descendente de <strong>25 niveles de presión</strong> entre <strong>1000 hPa</strong> y <strong>100 hPa</strong>, incluidos ambos extremos. Calcula las diferencias entre niveles consecutivos y su media, conservando el signo. Puedes utilizar <code>np.diff()</code>, que resta a cada elemento el anterior.<br><br>Explica si un espaciado uniforme en presión implica también un espaciado uniforme en altitud y relaciona tu respuesta con la estructura vertical de la atmósfera.</p>
</div>

In [58]:
presiones_des = np.linspace(1000, 100, 25)
print(presiones_des)

medias = np.mean(presiones_des)
print(medias)
diferencias = np.diff(presiones_des)
print(diferencias)

#presiones_des[0]

[1000.   962.5  925.   887.5  850.   812.5  775.   737.5  700.   662.5
  625.   587.5  550.   512.5  475.   437.5  400.   362.5  325.   287.5
  250.   212.5  175.   137.5  100. ]
550.0
[-37.5 -37.5 -37.5 -37.5 -37.5 -37.5 -37.5 -37.5 -37.5 -37.5 -37.5 -37.5
 -37.5 -37.5 -37.5 -37.5 -37.5 -37.5 -37.5 -37.5 -37.5 -37.5 -37.5 -37.5]


<div style="padding: 0.8em 1em; margin: 0.8em 0 1em; border-left: 4px solid #4C78A8; background-color: rgba(76,120,168,0.08); border-radius: 4px; line-height: 1.6;">
<p style="margin: 0.45em 0;"><strong>4. Desviaciones de precipitación y relleno de huecos</strong><br>Crea una copia de <code>precipitacion_mm</code> y sustituye los <code>np.nan</code> por la media de los meses con datos de esa misma serie. Después resta esa media a cada elemento para obtener sus desviaciones respecto de la referencia utilizada. Conserva intacto el array original y expresa los resultados en milímetros.</p>
</div>

<div style="padding: 0.8em 1em; margin: 0.8em 0 1em; border-left: 3px solid #72A0C1; background-color: rgba(76,120,168,0.025); border-radius: 4px; line-height: 1.6;">
<p style="margin: 0.45em 0;">Este relleno sirve para practicar la selección y la asignación. No reconstruye las precipitaciones desconocidas ni tiene en cuenta el ciclo estacional. Tampoco disponemos de una climatología de cada mes para calcular anomalías climatológicas.</p>
</div>

In [66]:
p_no_nan = precipitacion_mm.copy() 
media_precipitacion_sin_nan = np.nanmean(precipitacion_mm)
p_no_nan[np.isnan(p_no_nan)] = media_precipitacion_sin_nan
print(p_no_nan)


[45.2  38.   32.74 22.5  15.   10.1   5.3  32.74 30.2  48.7  52.4  60.  ]


<div style="padding: 0.8em 1em; margin: 0.8em 0 1em; border-left: 4px solid #4C78A8; background-color: rgba(76,120,168,0.08); border-radius: 4px; line-height: 1.6;">
<p style="margin: 0.45em 0;"><strong>5. Conversión de unidades de viento</strong><br>Utiliza el array <code>velocidad_viento</code> para convertir las velocidades de metros por segundo a nudos mediante la relación <code>1 m/s ≈ 1.94384 kt</code>. Identifica los días con velocidades estrictamente superiores a <strong>20 kt</strong>, numerando los días a partir de uno. Interpreta también el resultado si no hay ningún día que cumpla la condición.</p>
</div>